In [ ]:
r!pip install moviepy

!pip3 install imageio==2.4.1

!pip install --upgrade imageio-ffmpeg

!pip install tensorflow --upgrade

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 72.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for imageio: filename=imageio-2.4.1-py3-none-any.whl size=3303886 sha256=f56c68151cba87ad3537ca94bd346cf75286f6ca642ac15d6c00a4222d603b9c
  Stored in directory: /root/.cache/pip/wheels/be/7b/04/4d8d56f1d503e5c404f0de6018c0cfa592c71588a39b49e002
Successfully built imageio
  Attempting uninstall: imageio
    Found existing installation: imageio 2.9.0
    Uninstalling imageio-2.9.0:
      Successfully uninstalled imageio-2.9.0
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.9/26.9 MB 25.5 MB/s eta 0:00:00
Looking in indexes: https://pypi.org/simple, https://us-py

In [ ]:
from tensorflow.keras.applications.inception_resnet_v2 import InceptionResNetV2,preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image
import numpy as np
import pandas as pd
import cv2
import os
from scipy import spatial
from imutils.video import FileVideoStream
import imutils
import tensorflow as tf
from moviepy.video.io.ffmpeg_tools import ffmpeg_extract_subclip
from moviepy.editor import *
import matplotlib.pyplot as plt
from natsort import natsorted

Imageio: 'ffmpeg-linux64-v3.3.1' was not found on your computer; downloading it now.
Try 1. Download from https://github.com/imageio/imageio-binaries/raw/master/ffmpeg/ffmpeg-linux64-v3.3.1 (43.8 MB)
Downloading: 8192/45929032 bytes (0.0%)2793472/45929032 bytes (6.1%)6971392/45929032 bytes (15.2%)11083776/45929032 bytes (24.1%)15228928/45929032 bytes (33.2%)19251200/45929032 bytes (41.9%)23412736/45929032 bytes (51.0%)27303936/45929032 bytes (59.4%)31342592/45929032 bytes (68.2%)35504128/45929032 bytes (77.3%)39411712/45929032 bytes (85.8%)42835968/45929032 bytes (93.3%)45929032/45929032 bytes (100.0%)
  Done
File saved as /root

In [ ]:
def return_image_embedding1(model,img_path):
    img = image.load_img(img_path)
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)
    preds = model.predict(x)
    return preds

In [ ]:
def return_image_embedding2(model,img):
    # img = np.resize(frame, (299,299,3))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)
    preds = model.predict(x)
    print(preds.shape)
    return preds

In [ ]:
model = InceptionResNetV2(include_top=False,weights='imagenet',pooling='avg')

219055592/219055592 [==============================] - 11s 0us/step


In [ ]:
def cp_hist_to_cv(np_histogram_output):
    counts, bin_edges = np_histogram_output
    counts = cp.asnumpy(counts)
    return counts.ravel().astype('float32')

In [ ]:
from imutils.video import FileVideoStream
import imutils
import time
import numpy as np
import cupy as cp
import cv2
import math

vs = FileVideoStream('/content/drive/MyDrive/FootballData/Lalega/RAY_vs_ELC_-_Full_Match_-_Premier_Sport.mp4').start()

frame1 = vs.read()

j = 0

diction1 = {}
diction2 = {}

a = 0
ls1 = []
ls2 = []

while True:


  if not vs.more():
      vs.stop()
      break

  a += 1

  frame2 = vs.read()
  
  if frame2 is not None:

    hist_img1 = cp.histogram(cp.array(frame1))
    # np.normalize(hist_img1, hist_img1, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX)
    hist_img2 = cp.histogram(cp.array(frame2))
    # cv2.normalize(hist_img2, hist_img2, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX)

    # metric_val = bhattacharyya(cp_hist_to_cv(hist_img1),cp_hist_to_cv(hist_img2))
    metric_val = cv2.compareHist(cp_hist_to_cv(hist_img1), cp_hist_to_cv(hist_img2),
                        cv2.HISTCMP_HELLINGER)

    # print(metric_val)
    if metric_val >= 0.1:
      # k = 1
      # cv2.imwrite("/content/drive/MyDrive/FootballData/Camera Angel/" + str(j+1) +  str(k) +".jpeg",frame1)
      # k = 2
      # cv2.imwrite("/content/drive/MyDrive/FootballData/Camera Angel/" + str(j+1) + str(k) + ".jpeg",frame2)
      ls1.append(frame1)
      ls1.append(a//25)
      ls2.append(frame2)
      ls2.append(a//25)
      diction1["/content/drive/MyDrive/FootballData/Camera Angle1/" + str(j+1) + ".jpeg"] = ls1
      diction2["/content/drive/MyDrive/FootballData/Camera Angle2/" + str(j+1) + ".jpeg"] = ls2

 
      ls1 = []
      ls2 = []


      j += 1


    if a % 10000 == 0:
      print(a)

    frame1 = frame2


10000
20000
30000
40000
50000
60000
70000
80000
90000
100000
110000
120000
130000
140000
150000
160000


In [ ]:
print(len(diction1))
print(len(diction2))

1188
1188


In [ ]:
from time import sleep
import matplotlib.pyplot as plt

for i in diction1.keys():
  cv2.imwrite(i,diction1[i][0])
  cv2.imwrite(i.replace("Angle1","Angle2"),diction2[i.replace("Angle1","Angle2")][0])

In [ ]:
import os
path = "/content/drive/MyDrive/FootballData/Camera Angle1/"
files = sorted(os.listdir("/content/drive/MyDrive/FootballData/Camera Angle1"))
frametest1 = return_image_embedding1(model,"1.png")
frametest2 = return_image_embedding1(model,"2.jpeg")
frametest3 = return_image_embedding1(model,"3.jpeg")
frametest4 = return_image_embedding1(model,"4.jpeg")
ls1 = []
for i in files:
  frame2 = return_image_embedding1(model,path + i)
  similarity1 = -1 * (spatial.distance.cosine(frametest1, frame2) - 1)
  similarity2= -1 * (spatial.distance.cosine(frametest2, frame2) - 1)
  similarity3 = -1 * (spatial.distance.cosine(frametest3, frame2) - 1)
  similarity4 = -1 * (spatial.distance.cosine(frametest4, frame2) - 1)

  if similarity1 >= 0.9 or similarity2 >= 0.9 or similarity3 >= 0.9 or similarity4 >= 0.9:
    ls1.append(i)
    print(similarity1,   "-----------"  ,i)
    print(similarity2,   "-----------"  ,i)
    print(similarity3,   "-----------"  ,i)
    print(similarity4,   "-----------"  ,i)



1/1 [==============================] - 0s 55ms/step
0.9193293452262878 ----------- 10.jpeg
0.9340959787368774 ----------- 10.jpeg
0.9150274395942688 ----------- 10.jpeg
0.905807375907898 ----------- 10.jpeg
1/1 [==============================] - 0s 50ms/step
0.9224323630332947 ----------- 1003.jpeg
0.8636154532432556 ----------- 1003.jpeg
0.8760071396827698 ----------- 1003.jpeg
0.8852891325950623 ----------- 1003.jpeg
1/1 [==============================] - 0s 55ms/step
0.9026715159416199 ----------- 1010.jpeg
0.8841949105262756 ----------- 1010.jpeg
0.8553341627120972 ----------- 1010.jpeg
0.8486754298210144 ----------- 1010.jpeg
1/1 [==============================] - 0s 56ms/step
0.9318078756332397 ----------- 1011.jpeg
0.9182527661323547 ----------- 1011.jpeg
0.9215197563171387 ----------- 1011.jpeg
0.91793292760849 ----------- 1011.jpeg
1/1 [==============================] - 0s 50ms/step
0.9076710343360901 ----------- 1013.jpeg
0.862920343875885 ----------- 1013.jpeg
0.859165132045

In [ ]:
ls1 = natsorted(ls1)

In [ ]:
len(ls1)

127

In [ ]:
ls1

['10.jpeg',
 '94.jpeg',
 '111.jpeg',
 '117.jpeg',
 '136.jpeg',
 '143.jpeg',
 '145.jpeg',
 '147.jpeg',
 '149.jpeg',
 '159.jpeg',
 '161.jpeg',
 '180.jpeg',
 '188.jpeg',
 '204.jpeg',
 '220.jpeg',
 '222.jpeg',
 '224.jpeg',
 '231.jpeg',
 '233.jpeg',
 '243.jpeg',
 '246.jpeg',
 '248.jpeg',
 '281.jpeg',
 '300.jpeg',
 '303.jpeg',
 '313.jpeg',
 '316.jpeg',
 '318.jpeg',
 '341.jpeg',
 '356.jpeg',
 '358.jpeg',
 '373.jpeg',
 '376.jpeg',
 '378.jpeg',
 '391.jpeg',
 '394.jpeg',
 '396.jpeg',
 '403.jpeg',
 '423.jpeg',
 '433.jpeg',
 '435.jpeg',
 '438.jpeg',
 '446.jpeg',
 '450.jpeg',
 '461.jpeg',
 '462.jpeg',
 '463.jpeg',
 '487.jpeg',
 '491.jpeg',
 '498.jpeg',
 '512.jpeg',
 '518.jpeg',
 '569.jpeg',
 '578.jpeg',
 '590.jpeg',
 '591.jpeg',
 '619.jpeg',
 '633.jpeg',
 '638.jpeg',
 '647.jpeg',
 '649.jpeg',
 '651.jpeg',
 '662.jpeg',
 '664.jpeg',
 '666.jpeg',
 '669.jpeg',
 '677.jpeg',
 '685.jpeg',
 '691.jpeg',
 '693.jpeg',
 '698.jpeg',
 '700.jpeg',
 '702.jpeg',
 '709.jpeg',
 '711.jpeg',
 '713.jpeg',
 '716.jpeg',
 

In [ ]:
import shutil
path1 = "/content/drive/MyDrive/FootballData/Camera Angle1/"
path2 = "/content/drive/MyDrive/FootballData/Temp/"
for i in range(len(files)):
  if files[i] in ls1:
    try:
      shutil.copy(path1 + files[i],path2)

    except:
      continue
  else:
      continue

In [ ]:
import shutil
path1 = "/content/drive/MyDrive/FootballData/Camera Angle2/"
path2 = "/content/drive/MyDrive/FootballData/Temp2/"
for i in range(len(files)):
  if files[i] in temp:
    try:
      shutil.copy(path1 + files[i],path2)
    except:
      continue
  else:
      continue

Remove Files

In [ ]:
import shutil
path1 = "/content/drive/MyDrive/FootballData/Camera Angle1/"
for i in range(len(files)):
  if files[i] in ls1:
    continue
  else:
    try:
      os.remove(path1 + files[i])
    except:
      continue

In [ ]:
import os
path = "/content/drive/MyDrive/FootballData/Temp2/"
files = sorted(os.listdir("/content/drive/MyDrive/FootballData/Camera Angle2"))
frame1 = return_image_embedding1(model,"1.png")
ls2 = []
for i in files:
  frame2 = return_image_embedding1(model,path + i)
  similarity = -1 * (spatial.distance.cosine(frame1, frame2) - 1)
  if similarity > 0.8:
    ls2.append(i)
    print(similarity,   "-----------"  ,i)

1/1 [==============================] - 0s 21ms/step
0.9702714681625366 ----------- 1001.jpeg
1/1 [==============================] - 0s 24ms/step
0.9393628835678101 ----------- 1031.jpeg
1/1 [==============================] - 0s 21ms/step
0.8228703141212463 ----------- 1061.jpeg
1/1 [==============================] - 0s 21ms/step
0.9883657693862915 ----------- 1065.jpeg
1/1 [==============================] - 0s 24ms/step
0.9848796129226685 ----------- 1084.jpeg
1/1 [==============================] - 0s 21ms/step
0.8786640167236328 ----------- 154.jpeg
1/1 [==============================] - 0s 25ms/step
0.9381960034370422 ----------- 158.jpeg
1/1 [==============================] - 0s 22ms/step
0.9795351028442383 ----------- 195.jpeg
1/1 [==============================] - 0s 20ms/step
0.9836806654930115 ----------- 198.jpeg
1/1 [==============================] - 0s 23ms/step
0.8477927446365356 ----------- 199.jpeg
1/1 [==============================] - 0s 22ms/step
0.9081770181655884 ----

In [ ]:
path1 = "/content/drive/MyDrive/FootballData/Camera Angle2/"
# path2 = "/content/drive/MyDrive/FootballData/Camera Angle2/"
for i in range(len(files)):
  if files[i] in ls2:
    continue
  else:
    try:
      os.remove(path1 + files[i])

      # os.remove(path2 + files[i])
    except:
      continue

In [ ]:
video_dictionary = {}
for i in os.listdir("/content/drive/MyDrive/FootballData/Camera Angle1/"):
  video_dictionary["/content/drive/MyDrive/FootballData/Camera Angle1/" + i.replace(".jpeg",".mp4")] = diction1["/content/drive/MyDrive/FootballData/Camera Angle1/" + i] 

In [ ]:
vid = {}
for i in video_dictionary.keys():
  vid[i.replace("Camera Angle1","Videos")] = video_dictionary[i]

In [ ]:
vid.keys()

dict_keys(['/content/drive/MyDrive/FootballData/Videos/94.mp4', '/content/drive/MyDrive/FootballData/Videos/117.mp4', '/content/drive/MyDrive/FootballData/Videos/143.mp4', '/content/drive/MyDrive/FootballData/Videos/231.mp4', '/content/drive/MyDrive/FootballData/Videos/243.mp4', '/content/drive/MyDrive/FootballData/Videos/246.mp4', '/content/drive/MyDrive/FootballData/Videos/248.mp4', '/content/drive/MyDrive/FootballData/Videos/446.mp4', '/content/drive/MyDrive/FootballData/Videos/591.mp4', '/content/drive/MyDrive/FootballData/Videos/700.mp4', '/content/drive/MyDrive/FootballData/Videos/711.mp4', '/content/drive/MyDrive/FootballData/Videos/713.mp4', '/content/drive/MyDrive/FootballData/Videos/767.mp4', '/content/drive/MyDrive/FootballData/Videos/828.mp4', '/content/drive/MyDrive/FootballData/Videos/10.mp4', '/content/drive/MyDrive/FootballData/Videos/111.mp4', '/content/drive/MyDrive/FootballData/Videos/136.mp4', '/content/drive/MyDrive/FootballData/Videos/145.mp4', '/content/drive/MyD

In [ ]:
len(vid)

127

In [ ]:
# cutting out some part from the clip
for i in vid.keys():
  ffmpeg_extract_subclip("/content/drive/MyDrive/FootballData/Lalega/RAY_vs_ELC_-_Full_Match_-_Premier_Sport.mp4", vid[i][1]-10, vid[i][1]+10, targetname=i)



[MoviePy] Running:
>>> /usr/bin/ffmpeg -y -i /content/drive/MyDrive/FootballData/Lalega/RAY_vs_ELC_-_Full_Match_-_Premier_Sport.mp4 -ss 111.00 -t 20.00 -vcodec copy -acodec copy /content/drive/MyDrive/FootballData/Videos/94.mp4
... command successful.

[MoviePy] Running:
>>> /usr/bin/ffmpeg -y -i /content/drive/MyDrive/FootballData/Lalega/RAY_vs_ELC_-_Full_Match_-_Premier_Sport.mp4 -ss 229.00 -t 20.00 -vcodec copy -acodec copy /content/drive/MyDrive/FootballData/Videos/117.mp4
... command successful.

[MoviePy] Running:
>>> /usr/bin/ffmpeg -y -i /content/drive/MyDrive/FootballData/Lalega/RAY_vs_ELC_-_Full_Match_-_Premier_Sport.mp4 -ss 369.00 -t 20.00 -vcodec copy -acodec copy /content/drive/MyDrive/FootballData/Videos/143.mp4
... command successful.

[MoviePy] Running:
>>> /usr/bin/ffmpeg -y -i /content/drive/MyDrive/FootballData/Lalega/RAY_vs_ELC_-_Full_Match_-_Premier_Sport.mp4 -ss 927.00 -t 20.00 -vcodec copy -acodec copy /content/drive/MyDrive/FootballData/Videos/231.mp4
... comma

In [ ]:
# ls = []
# for i in os.listdir("/content/drive/MyDrive/FootballData/Videos/"):
#   video = VideoFileClip("/content/drive/MyDrive/FootballData/Videos/" + i)
#   ls.append(video)
# final_clip = concatenate_videoclips(ls)
# final_clip.to_videofile("/content/drive/MyDrive/FootballData/Videos/output.mp4", fps=25, remove_temp=False)

[MoviePy] >>>> Building video /content/drive/MyDrive/FootballData/Videos/output.mp4
[MoviePy] Writing audio in outputTEMP_MPY_wvf_snd.mp3


100%|██████████| 14334/14334 [00:25<00:00, 560.73it/s]

[MoviePy] Done.
[MoviePy] Writing video /content/drive/MyDrive/FootballData/Videos/output.mp4



  3%|▎         | 453/16251 [00:21<12:17, 21.43it/s]


KeyboardInterrupt: ignored

In [ ]:
len(vid)

110

In [ ]:
ls = []
import matplotlib.pyplot as plt
backSub = cv2.createBackgroundSubtractorMOG2()

capture = cv2.VideoCapture('/content/drive/MyDrive/FootballData/Premier League/ARS_vs_TOT_-_Second-Half.mp4')
if not capture.isOpened():
    print('Unable to open: ')
    exit(0)
while True:
    ret, frame = capture.read()
    if frame is None:
        break
    
    fgMask = backSub.apply(frame)
    
    
    
    
    plt.imshow(frame)


    plt.imshow(fgMask)
    ls.append(fgMask)
    
    keyboard = cv2.waitKey(30)
    if keyboard == 'q' or keyboard == 27:
        break

KeyboardInterrupt: ignored

Error in callback <function install_repl_displayhook.<locals>.post_execute at 0x7f36558e4160> (for post_execute):


KeyboardInterrupt: ignored

Error in callback <function flush_figures at 0x7f3655685d30> (for post_execute):


KeyboardInterrupt: ignored

In [ ]:
plt.imshow(ls[0])

In [ ]:
# i = 0
# m = 0
# temp = []
# while ( i < len(ls1)):
#   j = int(ls1[i].replace(".jpeg",""))

#   a = 0
#   for k in range(j+1,j+20):
#     if str(k) + ".jpeg" in ls1:
#       pass

#     elif str(k) + ".jpeg" not in ls1:
#       m = k-1
#       i += a

#       break

#     a += 1

#   temp.append(str(m) + ".jpeg")
#   i += 1


In [ ]:
import matplotlib.pyplot as plt
plt.imshow(ls[])

IndexError: ignored

In [ ]:
import gc

gc.collect()

49909